# Estimating EM sovereign risk premia

## Get packages and JPMaQS data

### Setup and imports

In [ ]:
# !pip install macrosynergy --upgrade

In [1]:
# Constants and credentials
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file in parent directory
env_path = Path(__file__).parent.parent / '.env' if '__file__' in globals() else Path.cwd().parent / '.env'
load_dotenv(dotenv_path=env_path)

REQUIRED_VERSION: str = "1.2.2"
DQ_CLIENT_ID: str = os.getenv("DQ_CLIENT_ID")
DQ_CLIENT_SECRET: str = os.getenv("DQ_CLIENT_SECRET")
PROXY = {}  # Configure if behind corporate firewall
START_DATE: str = "1998-01-01"

# Verify credentials loaded
if not DQ_CLIENT_ID or not DQ_CLIENT_SECRET:
    raise ValueError("DQ_CLIENT_ID and DQ_CLIENT_SECRET must be set in .env file")
    
print(f"✓ Credentials loaded successfully (Client ID: {DQ_CLIENT_ID})")
print(f"✓ .env file loaded from: {env_path}")

✓ Credentials loaded successfully (Client ID: vargas31)
✓ .env file loaded from: c:\code\em_debt\.env


In [2]:
import macrosynergy as msy

msy.check_package_version(required_version=REQUIRED_VERSION)
# If version check fails: pip install macrosynergy --upgrade

c:\code\em_debt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import numpy as np
import pandas as pd
from pandas import Timestamp
import matplotlib.pyplot as plt
import os
import io

from datetime import datetime

import macrosynergy.management as msm
import macrosynergy.panel as msp
import macrosynergy.signal as mss
import macrosynergy.pnl as msn
import macrosynergy.visuals as msv

from macrosynergy.download import JPMaQSDownload
from macrosynergy.panel.adjust_weights import adjust_weights
import warnings

pd.set_option('display.width', 400)

warnings.simplefilter("ignore")

### Data selection and download

In [4]:
cids_fc_latam = [  # Latam foreign currency debt countries
    "BRL",
    "CLP",
    "COP",
    "DOP",
    "MXN",
    "PEN",
    "UYU",
]
cids_fc_emeu = [  # EM Europe foreign currency debt countries
    "HUF",
    "PLN",
    "RON",
    "RUB",
    "RSD",
    "TRY",
]
cids_fc_meaf = [  # Middle-East and Africa foreign currency debt countries
    "AED",
    "EGP",
    "NGN",
    "OMR",
    "QAR",
    "ZAR",
    "SAR",
]
cids_fc_asia = [  # Asia foreign currency debt countries
    "CNY",
    "IDR",
    "INR",
    "PHP",
]

cids_fc = sorted(
    list(
        set(cids_fc_latam + cids_fc_emeu + cids_fc_meaf + cids_fc_asia)
    )
)
cids_emxfc = ["CZK", "ILS", "KRW", "MYR", "SGD", "THB", "TWD"]

cids_em = sorted(cids_fc + cids_emxfc)

In [5]:
# Category tickers

# Features
govfin = [
    "GGOBGDPRATIO_NSA",
    "GGOBGDPRATIONY_NSA",
    "GGDGDPRATIO_NSA",
]

xbal = [
    "CABGDPRATIO_NSA_12MMA",
    "MTBGDPRATIO_NSA_12MMA",
]

xliab = [
    "NIIPGDP_NSA_D1Mv2YMA",
    "NIIPGDP_NSA_D1Mv5YMA",
    "IIPLIABGDP_NSA_D1Mv2YMA",
    "IIPLIABGDP_NSA_D1Mv5YMA",
]

xdebt = [
    "ALLIFCDSGDP_NSA",
    "GGIFCDSGDP_NSA",
]

GOVRISK = [
    "ACCOUNTABILITY_NSA",
    "POLSTAB_NSA",
    "CORRCONTROL_NSA",
]
growth = [
    'RGDP_SA_P1Q1QL4_20QMA',
    "RGDP_SA_P1Q1QL4"
] 

infl = [
    "CPIC_SA_P1M1ML12",
    "CPIH_SA_P1M1ML12",
]
 

risk_metrics = [
    "LTFCRATING_NSA",
    "LTLCRATING_NSA",
    "FCBICRY_NSA",
    "FCBICRY_VT10",
    "CDS05YSPRD_NSA",
    "CDS05YXRxEASD_NSA",
]

# Targets

rets = ["FCBIR_NSA", "FCBIXR_NSA", "FCBIXR_VT10"]

# Benchmark returns

bms = ["USD_EQXR_NSA", "UHY_CRXR_NSA", "UIG_CRXR_NSA"]

# Create ticker list

xcats = govfin + xbal + xliab + xdebt + GOVRISK + growth + infl + risk_metrics + rets
tickers = [cid + "_" + xcat for cid in cids_em for xcat in xcats] + bms

print(f"Maximum number of tickers is {len(tickers)}")

Maximum number of tickers is 840


In [6]:
# Download macro-quantamental indicators from JPMaQS via the DataQuery API

with JPMaQSDownload(
    client_id=DQ_CLIENT_ID, 
    client_secret=DQ_CLIENT_SECRET,
    proxy=PROXY
) as downloader:
    df: pd.DataFrame = downloader.download(
        tickers=tickers,
        start_date=START_DATE,
        metrics=["value",],
        suppress_warning=True,
        show_progress=True,
        report_time_taken=True,
        get_catalogue=True,
    )

AuthenticationError: AuthenticationError: Error retrieving token: 401 Client Error: Unauthorized for url: https://authe.jpmchase.com/as/token.oauth2 
AuthenticationError

In [ ]:
# Preserve original downloaded data for debugging and comparison
dfx = df.copy()
dfx.info()

NameError: name 'df' is not defined

## Availability checks and blacklisting

### Availability

In [ ]:
xcatx = govfin
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = xbal
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = xliab
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = xdebt
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = risk_metrics
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = rets
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = growth
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

In [ ]:
xcatx = infl
msm.check_availability(df=dfx, xcats=xcatx, cids=cids_em, missing_recent=False)

### Blacklist

In [ ]:
black_fc = {'RUB': [Timestamp('2022-02-01 00:00:00'), Timestamp('2035-02-26 00:00:00')]}

## Macro risk score construction and checks

### Preparatory transformations

In [ ]:
# Non-linear inflation effect theory

x = np.linspace(-5, 100, 100)
y = np.power(abs(x - 2), 1/2)

plt.figure(figsize=(12, 4))
plt.plot(x, y)
plt.title("Inflation and presumed effect on sovereign default risk")
plt.xlabel("Inflation")
plt.ylabel("Presumed impact on default risk")
plt.grid(True)
plt.show()

In [ ]:
# Inflation effects

cidx = cids_fc
xcx = ["CPIH", "CPIC"]

calcs = [
    f"{xc}_IE = {xc}_SA_P1M1ML12.applymap(lambda x: np.power(abs(x - 2), 1/2))"
    for xc in xcx
]
dfa = msp.panel_calculator(dfx, calcs=calcs, cids=cidx)
dfx = msm.update_df(dfx, dfa)


xcatx = ["CPIH_IE", "CPIH_SA_P1M1ML12"]


msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start="2000-01-01",
    title='CPI comparison - square function vs standard inflation change',
    aspect=2,
    same_y=False,
)

### Macro risk scores

#### Conceptual factor scores

In [ ]:
# Placeholder dictionaries

dict_factz = {}
dict_labels = {}

In [ ]:
# Governing dictionary of quantamental indicators, neutral levels, and weightings

dict_macros = {
    "GFINRISK":{
        "GGOBGDPRATIO_NSA": ["median", "NEG", 1/3],
        "GGOBGDPRATIONY_NSA": ["median", "NEG", 1/3],
        "GGDGDPRATIO_NSA": ["median", "", 1/3],
    },
    "XBALRISK":{
        "CABGDPRATIO_NSA_12MMA": ["median", "NEG", 0.5],
        "MTBGDPRATIO_NSA_12MMA": ["median", "NEG", 0.5],
    },
    "XLIABRISK":{
        "NIIPGDP_NSA_D1Mv2YMA": ["median", "NEG", 0.25],
        "NIIPGDP_NSA_D1Mv5YMA": ["median", "NEG", 0.25],
        "IIPLIABGDP_NSA_D1Mv2YMA": ["median", "", 0.25],
        "IIPLIABGDP_NSA_D1Mv5YMA": ["median", "", 0.25],
    },
    "XDEBTRISK":{
        "ALLIFCDSGDP_NSA": ["median", "", 0.5],
        "GGIFCDSGDP_NSA": ["median", "", 0.5],
    },
    "GOVRISK":{
        "ACCOUNTABILITY_NSA": ["median", "NEG", 1/3],
        "POLSTAB_NSA": ["median", "NEG", 1/3],
        "CORRCONTROL_NSA": ["median", "NEG", 1/3],
    },
    "GROWTHRISK":{
        "RGDP_SA_P1Q1QL4_20QMA": ["median", "NEG", 0.5],
        "RGDP_SA_P1Q1QL4": ["median", "NEG", 0.5],
    },
    "INFLRISK":{
        "CPIH_IE": ["median", "", 0.5],
        "CPIC_IE": ["median", "", 0.5],
    }
}

In [ ]:
# Normalize all macro-quantamental categories based on the broad EM set

cidx = cids_em

for fact in dict_macros.keys():
    
    dict_fact = dict_macros[fact]
    xcatx = list(dict_fact.keys())

    dfa = pd.DataFrame(columns=list(dfx.columns))

    for xc in xcatx:

        postfix = dict_fact[xc][1]
        neutral = dict_fact[xc][0]

        dfaa = msp.make_zn_scores(
            dfx,
            xcat=xc,
            cids=cidx,
            sequential=True,
            min_obs=261 * 3,
            neutral=neutral,
            pan_weight=1,
            thresh=3,
            postfix="_" + postfix + "ZN",
            est_freq="m",
        )
        dfaa["value"] = dfaa["value"] * (1 if postfix == "" else -1)
        dfa = msm.update_df(dfa, dfaa)

    dict_factz[fact] = dfa["xcat"].unique()
    dfx = msm.update_df(dfx, dfa)

In [ ]:
# Combine quantamental scores to conceptual scores

cidx = cids_em

for key, value in dict_factz.items():

    weights = [w[2] for w in list(dict_macros[key].values())]
    dfa = msp.linear_composite(
        dfx,
        xcats=value,
        cids=cidx,
        weights=weights,
        normalize_weights= True,
        complete_xcats=False,
        new_xcat=key,
    )

    dfx = msm.update_df(dfx, dfa)

for key in dict_factz.keys():

    dfa = msp.make_zn_scores(
        dfx,
        xcat=key,
        cids=cidx,
        sequential=True,
        min_obs=261 * 3,
        neutral="zero",
        pan_weight=1,
        thresh=3,
        postfix="_ZN",
        est_freq="m",
    )
    
    dfx = msm.update_df(dfx, dfa)

macroz = [fact + "_ZN" for fact in dict_factz.keys()]

In [ ]:
dict_labels["GGOBGDPRATIO_NSA_NEGZN"] = "Excess government balance, % of GDP, current year, negative"
dict_labels["GGOBGDPRATIONY_NSA_NEGZN"] = "Excess government balance, % of GDP, next year, negative"
dict_labels["GGDGDPRATIO_NSA_ZN"] = "Excess general government debt ratio, % of GDP"

dict_labels["GFINRISK_ZN"] = "Government finances risk score"

dict_labels["CABGDPRATIO_NSA_12MMA_NEGZN"] = "Current account balance, 12mma, % of GDP, negative"
dict_labels["MTBGDPRATIO_NSA_12MMA_NEGZN"] = "Merchandise trade balance, 12mma, % of GDP, negative"

dict_labels["XBALRISK_ZN"] = "External balances risk score"

dict_labels["IIPLIABGDP_NSA_D1Mv2YMA_ZN"] = "International liabilities, vs. 2yma, % of GDP"
dict_labels["IIPLIABGDP_NSA_D1Mv5YMA_ZN"] = "International liabilities, vs. 5yma, % of GDP"
dict_labels["NIIPGDP_NSA_D1Mv2YMA_NEGZN"] = "Net international investment position, vs. 2yma, % of GDP, negative"
dict_labels["NIIPGDP_NSA_D1Mv5YMA_NEGZN"] = "Net international investment position, vs. 5yma, % of GDP, negative"

dict_labels["XLIABRISK_ZN"] = "International position risk score"

dict_labels["ALLIFCDSGDP_NSA_NEGZN"] = "Excess foreign-currency debt securities, all, % of GDP, negative"
dict_labels["GGIFCDSGDP_NSA_NEGZN"] = "Excess foreign-currency debt securities, government, % of GDP, negative"

dict_labels["XDEBTRISK_ZN"] = "Foreign-currency debt risk score"

dict_labels["ACCOUNTABILITY_NSA_NEGZN"] = "Voice and accountability index, z-score"
dict_labels["POLSTAB_NSA_NEGZN"] = "Political stability and absence of violence/terrorism index, z-score"
dict_labels["CORRCONTROL_NSA_NEGZN"] = "Corruption control index, z-score"

dict_labels['GOVRISK_ZN'] = 'Governance risk score'

dict_labels["RGDP_SA_P1Q1QL4_20QMA_NEGZN"] = "Real GDP growth, percentage over a year ago, z-score, negative"
dict_labels["RGDP_SA_P1Q1QL4_NEGZN"] = "Real GDP growth, percentage over a year ago, z-score"

dict_labels['GROWTHRISK_ZN'] = "Growth risk score"

dict_labels["CPIH_IE_NEGZN"] = "Headline CPI inflation effect, z-score"
dict_labels["CPIC_IE_NEGZN"] = "Core CPI inflation effect, z-score"

dict_labels["INFLRISK_ZN"] = "Inflation risk score"

In [ ]:
dict_countries = {
    'AED': 'United Arab Emirates',
    'BRL': 'Brazil',
    'CLP': 'Chile',
    'CNY': 'China',
    'COP': 'Colombia',
    'DOP': 'Dominican Republic',
    'EGP': 'Egypt',
    'HUF': 'Hungary',
    'IDR': 'Indonesia',
    'INR': 'India',
    'MXN': 'Mexico',
    'NGN': 'Nigeria',
    'OMR': 'Oman',
    'PEN': 'Peru',
    'PHP': 'Philippines',
    'PLN': 'Poland',
    'QAR': 'Qatar',
    'RON': 'Romania',
    'RSD': 'Serbia',
    'RUB': 'Russia',
    'SAR': 'Saudi Arabia',
    'TRY': 'Turkey',
    'UYU': 'Uruguay',
    'ZAR': 'South Africa'
}

In [ ]:
# Box for quantamental score review

factor = "XBALRISK"   # "GFINRISK" "XBALRISK" "XLIABRISK" "XDEBTRISK" "GOVRISK" "GROWTHRISK" "INFLRISK"
xcatx = list(dict_factz[factor])
cidx = cids_fc
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by="mean",  # countries sorted by mean of the first category
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title=None,
    title_fontsize=22,
    legend_fontsize=16,
    height=2,
)

In [ ]:
xcatx = macroz
cidx = cids_fc
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title="Macro scores related to sovereign debt default risk (higher score means higher risk)",
    title_fontsize=25,
    legend_fontsize=16,
    height=2,
)

#### Composite macro risk scores

In [ ]:
# Custom weights

dict_custom_weights = {
    'GFINRISK_ZN': 1,
    'XBALRISK_ZN': 1,
    'XLIABRISK_ZN': 1,
    'XDEBTRISK_ZN': 0.001,
    'GOVRISK_ZN': 1,
    'GROWTHRISK_ZN': 0.001,
    'INFLRISK_ZN': 0.001,
}

reduced_dict_custom_weights = {
    'GFINRISK_ZN': 1,
    'XBALRISK_ZN': 1,
    'XLIABRISK_ZN': 1,
    'GOVRISK_ZN': 1
}


In [ ]:
# Weighted composite macro risk scores

cidx = cids_fc
xcatx = macroz

reduced_macroz = ['GFINRISK_ZN', 'XBALRISK_ZN', 'XLIABRISK_ZN', 'GOVRISK_ZN']

equal_weights = [1/len(macroz)] * len(macroz)
reduced_custom_weights = [reduced_dict_custom_weights[m] for m in reduced_macroz]

weights = {
    "EWS": (macroz, equal_weights),
    "CWS": (reduced_macroz, reduced_custom_weights),
}

# Step 4: Loop through
for k, (xcat_list, weight_list) in weights.items():
    dfa = msp.linear_composite(
        dfx,
        xcats=xcat_list,
        cids=cids_fc,
        weights=weight_list,
        normalize_weights=True,
        complete_xcats=False,
        new_xcat="MACRORISK_" + k,
    )
    dfx = msm.update_df(dfx, dfa)

# Re-scoring the composites

for m in ['MACRORISK_EWS', 'MACRORISK_CWS']:
    dfa = msp.make_zn_scores(
        dfx,
        xcat=m,
        cids=cidx,
        sequential=True,
        min_obs=261 * 3,
        neutral="zero",
        pan_weight=1,
        thresh=3,
        postfix="_ZN",
        est_freq="m",
    )
    
    dfx = msm.update_df(dfx, dfa)

dict_labels['MACRORISK_EWS_ZN'] = 'Composite macro risk score'
dict_labels['MACRORISK_CWS_ZN'] = 'Custom weighted macro risk score'

In [ ]:
xcatx = ['MACRORISK_EWS_ZN'] # ['MACRORISK_EWS_ZN', 'MACRORISK_CWS_ZN']
cidx = cids_fc
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title="Composite macro risk related to sovereign risk, equally-weighted score of 7 constituents",
    title_fontsize=25,
    legend_fontsize=16,
    height=2,
)

## Market-implied risk scores

In [ ]:
# Use index carry where CDS spreads not available ("priced risk" score)

msm.missing_in_df(df, xcats=["CDS05YSPRD_NSA"], cids=cids_fc)  # countries without CDS
cidx = ['AED', 'DOP', 'EGP', 'INR', 'NGN', 'OMR', 'QAR', 'RSD', 'SAR', 'UYU']

calcs = ["CDS05YSPRD_NSA = FCBICRY_NSA"]
dfa = msp.panel_calculator(dfx, calcs=calcs, cids=cidx)
dfx = msm.update_df(dfx, dfa)

# Use inverse rating score ("rated risk" score)

calcs = [
    "LTFCRATING_ADJ = LTFCRATING_NSA + 1",  # temporary fix for VEF/RUB problem
    "LTFCRATING_INV = 1 / LTFCRATING_ADJ",
    "CDS05YSPRD_LOG = np.log( CDS05YSPRD_NSA )"  # accoount for non-linearity of spread changes
    
]
cidx = cids_fc
dfa = msp.panel_calculator(dfx, calcs=calcs, cids=cidx)
dfx = msm.update_df(dfx, dfa)

In [ ]:
xcatx = ["LTFCRATING_INV", "MACRORISK_EWS_ZN"]
cidx = cids_fc

catregs = {}

for xc in xcatx:
    catregs[xc] = msp.CategoryRelations(
        dfx,
        xcats=[xc, "CDS05YSPRD_NSA"],
        cids=cidx,
        years=1,
        lag=0,
        xcat_aggs=["mean", "mean"],
        blacklist=black_fc,
        start="2000-01-01",
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    reg_order=2,
    share_axes=False,
    ncol=2,
    nrow=1,
    figsize=(14, 7),
    title="Risk indices and sovereign credit spreads, annual averages, 24 countries, since 2000",
    title_fontsize=18,
    xlab="Risk index",
    ylab="Sovereign credit spread, %",
    subplot_titles=["Ratings-related risk index", "Macro risk score"],
)

In [ ]:
# Re-score the composite

for xc in ["CDS05YSPRD_LOG", "LTFCRATING_INV"]:
    dfa = msp.make_zn_scores(
        dfx,
        xcat=xc,
        cids=cidx,
        sequential=True,
        min_obs=261 * 3,
        neutral="median",
        pan_weight=1,
        thresh=3,
        postfix="_ZN",
        est_freq="m",
    )
    
    dfx = msm.update_df(dfx, dfa)

dict_labels["CDS05YSPRD_LOG_ZN"] = "Log spread-based market risk score"
dict_labels["LTFCRATING_INV_ZN"] = "Ratings-based market risk score"

In [ ]:
# Composite market risk scores

cidx = cids_fc
xcatx = ["CDS05YSPRD_LOG_ZN", "LTFCRATING_INV_ZN"]

dfa = msp.linear_composite(
    dfx,
    xcats=xcatx,
    cids=cidx,
    complete_xcats=False,
    new_xcat="MARKETRISK",
)

dfx = msm.update_df(dfx, dfa)

# Re-score the composite

dfa = msp.make_zn_scores(
    dfx,
    xcat="MARKETRISK",
    cids=cidx,
    sequential=True,
    min_obs=261 * 3,
    neutral="zero",
    pan_weight=1,
    thresh=3,
    postfix="_ZN",
    est_freq="m",
)

dfx = msm.update_df(dfx, dfa)

dict_labels["MARKETRISK_ZN"] = "Composite market risk score"

In [ ]:
xcatx = ["MARKETRISK_ZN", "CDS05YSPRD_LOG_ZN", "LTFCRATING_INV_ZN"]
cidx = cids_fc
sdate = "2000-01-01"


msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by="mean",
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    xcat_labels=dict_labels,
    title='Market risk scores for foreign-currency sovereign debt of major EMBI countries',
    title_fontsize=25,
    legend_fontsize=16,
    cid_labels=dict_countries,
    aspect=2,
    height=2,
    blacklist=black_fc
)

## Basic relationship visualizations

### Long-term relations

In [ ]:
# Long-term ratings - spread relations

xcatx = ["LTFCRATING_INV_ZN", "CDS05YSPRD_LOG_ZN"]
cidx = cids_fc

cr = msp.CategoryRelations(
    dfx,
    xcats=xcatx,
    cids=cidx,
    years=5,
    lag=0,
    xcat_aggs=["mean", "mean"],
    blacklist=black_fc,
    start="2000-01-01",
)

cr.reg_scatter(
    labels=True,
    label_fontsize=12,
    title="Long-term relations between credit spread and rated risk scores, by half-decades, since 2000",
    title_fontsize=16,
    xlab="Rated risk score, half-decade average",
    ylab="Credit spread score, half-decade average",
)

In [ ]:
# Long-term macro risk - spread relations

xcatx = ["MACRORISK_EWS_ZN", "CDS05YSPRD_LOG_ZN"]
cidx = cids_fc

cr = msp.CategoryRelations(
    dfx,
    xcats=xcatx,
    cids=cidx,
    years=5,
    lag=0,
    xcat_aggs=["mean", "mean"],
    blacklist=black_fc,
    start="2000-01-01",
)

cr.reg_scatter(
    labels=True,
    label_fontsize=12,
    title="Long-term relations between composite sovereign risk and credit spread scores, by half-decades, since 2000",
    title_fontsize=16,
    xlab="Sovereign credit-related macro risk score, half-decade average (as far as available)",
    ylab=dict_labels['CDS05YSPRD_LOG_ZN'],
)

In [ ]:
# Long-term macro risk - ratings relations

xcatx = ["MACRORISK_EWS_ZN", "LTFCRATING_INV_ZN"]
cidx = cids_fc

cr = msp.CategoryRelations(
    dfx,
    xcats=xcatx,
    cids=cidx,
    years=5,
    lag=0,
    xcat_aggs=["mean", "mean"],
    blacklist=black_fc,
    start="2000-01-01",
)

cr.reg_scatter(
    labels=True,
    label_fontsize=12,
    title="Long-term relations between composite sovereign risk and rated risk scores, by half-decades, since 2000",
    title_fontsize=16,
    xlab="Sovereign credit-related macro risk score, half-decade average (as far as available)",
    ylab=dict_labels['LTFCRATING_INV_ZN'],
)

In [ ]:
# Long-term macro risk - ratings relations

xcatx = ["MACRORISK_EWS_ZN", "MARKETRISK_ZN"]
cidx = cids_fc

cr = msp.CategoryRelations(
    dfx,
    xcats=xcatx,
    cids=cidx,
    years=5,
    lag=0,
    xcat_aggs=["mean", "mean"],
    blacklist=black_fc,
    start="2000-01-01",
)

cr.reg_scatter(
    labels=True,
    label_fontsize=12,
    title="EM credit risk: macro risk scores and market risk scores, by half-decades, 24 sovereigns, since 2000",
    title_fontsize=16,
    xlab="Sovereign credit-related macro risk score, half-decade average (as far as available)",
    ylab="Market risk score, half-decade average (as far as available)",
)

In [ ]:
cidx = cids_fc

risks = ["MARKETRISK_ZN", "MACRORISK_EWS_ZN"]
returns = ["FCBIXR_NSA", "FCBIXR_VT10"]

dict_labels["FCBIXR_NSA"] = "Foreign-currency sovereign bond index return, %"
dict_labels["FCBIXR_VT10"] = "Vol-targeted foreign-currency sovereign bond index return, %"

all_relations = []
all_titles = []

for risk in risks:
    for ret in returns:
        cr = msp.CategoryRelations(
            dfx,
            xcats=[risk, ret],
            cids=cidx,
            freq="A",
            # years=10,
            lag=1,
            xcat_aggs=["mean", "sum"],
            blacklist=black_fc,
            start="2000-01-01",
        )
        all_relations.append(cr)
        risk_label = dict_labels[risk]
        ret_label = dict_labels[ret].lower()
        all_titles.append(risk_label + " vs. " + ret_label)

msv.multiple_reg_scatter(
    cat_rels=all_relations,
    title="Annual risk scores and subsequent returns for foreign-currency sovereign debt, 24 countries, since 2000",
    xlab="Risk score (annual average)",
    ylab="Returns, %, next year",
    ncol=2,
    nrow=2,
    figsize=(14, 12),
    prob_est="map",
    coef_box="lower left",
    subplot_titles=all_titles
)

In [ ]:
cidx = cids_fc

risks = ["CDS05YSPRD_LOG_ZN", "LTFCRATING_INV_ZN"]
returns = ["FCBIXR_NSA", "FCBIXR_VT10"]

all_relations = []
all_titles = []

for risk in risks:
    for ret in returns:
        cr = msp.CategoryRelations(
            dfx,
            xcats=[risk, ret],
            cids=cidx,
            freq="A",
            # years=10,
            lag=1,
            xcat_aggs=["mean", "sum"],
            blacklist=black_fc,
            start="2000-01-01",
        )
        all_relations.append(cr)
        risk_label = dict_labels[risk]
        ret_label = dict_labels[ret].lower()
        all_titles.append(risk_label + " vs. " + ret_label)

msv.multiple_reg_scatter(
    cat_rels=all_relations,
    title=None,
    xlab=None,
    ylab=None,
    ncol=2,
    nrow=2,
    figsize=(16, 10),
    prob_est="map",
    coef_box="lower left",
    subplot_titles=all_titles
)

## Macro risk premium scores

### Aggregate macro risk premium scores

In [ ]:
# Score differences

cidx = cids_fc

calcs = [
    "MACROSPREAD_RPS = CDS05YSPRD_LOG_ZN - MACRORISK_EWS_ZN",
    "MACRORATING_RPS = LTFCRATING_INV_ZN - MACRORISK_EWS_ZN",
    "MACROALL_RPS = MARKETRISK_ZN - MACRORISK_EWS_ZN",
    "MACROSPREAD_RPS_CWS = CDS05YSPRD_LOG_ZN - MACRORISK_CWS_ZN",
    "MACRORATING_RPS_CWS = LTFCRATING_INV_ZN - MACRORISK_CWS_ZN",
    "MACROALL_RPS_CWS = MARKETRISK_ZN - MACRORISK_CWS_ZN",
]
dfa = msp.panel_calculator(dfx, calcs=calcs, cids=cidx)
dfx = msm.update_df(dfx, dfa)

rps = list(dfa['xcat'].unique())

# Re-z-scoring the risk premium scores

for rp in rps:
    dfa = msp.make_zn_scores(
        dfx,
        xcat=rp,
        cids=cidx,
        sequential=True,
        min_obs=261 * 3,
        neutral="zero",
        pan_weight=1,
        thresh=3,
        postfix="_ZN",
        est_freq="m",
    )
    dfx = msm.update_df(dfx, dfa)

dict_labels["MACROSPREAD_RPS_ZN"] = "Spread-based macro risk premium score"
dict_labels["MACRORATING_RPS_ZN"] = "Ratings-based macro risk premium score"
dict_labels["MACROALL_RPS_ZN"] = "Macro risk premium score"

dict_labels["MACROSPREAD_RPS_CWS_ZN"] = "Spread-based macro risk premium score, custom weights"
dict_labels["MACRORATING_RPS_CWS_ZN"] = "Ratings-based macro risk premium score, custom weights"
dict_labels["MACROALL_RPS_CWS_ZN"] = "Macro risk premium score, custom weights"

rpz = ["MACROSPREAD_RPS_ZN", "MACRORATING_RPS_ZN", "MACROALL_RPS_ZN",]
xrpz = rpz + [rp[:-3] + "_CWS_ZN" for rp in rpz]

In [ ]:
xcatx = rpz
cidx = cids_fc
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title="Sovereign-credit-related macro risk premium scores (higher score means higher premium)",
    title_fontsize=25,
    legend_fontsize=16,
    height=2,
)

### Conceptual macro risk premium scores

In [ ]:
# Conceptual risk premium scores

calcs = []
for macro in macroz:
    calcs += f"{macro[:-3]}_RPS = MARKETRISK_ZN - {macro}",

dfa = msp.panel_calculator(dfx, calcs=calcs, cids=cidx)
dfx = msm.update_df(dfx, dfa)

crps = list(dfa['xcat'].unique())

# Re-z-scoring conceptual risk premium scores

for crp in crps:
    dfa = msp.make_zn_scores(
        dfx,
        xcat=crp,
        cids=cidx,
        sequential=True,
        min_obs=261 * 3,
        neutral="zero",
        pan_weight=1,
        thresh=3,
        postfix="_ZN",
        est_freq="m",
    )
    dfx = msm.update_df(dfx, dfa)

crpz = [crp + "_ZN" for crp in crps]

dict_labels['GFINRISK_RPS_ZN'] = 'Government finance conceptual macro risk premium score'
dict_labels['XBALRISK_RPS_ZN'] = 'External balances conceptual macro risk premium score'
dict_labels['XLIABRISK_RPS_ZN'] = 'International position conceptual macro risk premium score'
dict_labels['XDEBTRISK_RPS_ZN'] = 'Foreign-currency debt conceptual macro risk premium score'
dict_labels['GOVRISK_RPS_ZN'] = 'Governance conceptual macro risk premium score'
dict_labels['GROWTHRISK_RPS_ZN'] = 'Growth risk conceptual macro risk premium score'
dict_labels['INFLRISK_RPS_ZN'] = 'Inflation risk conceptual macro risk premium score'

In [ ]:
xcatx = crpz
cidx = cids_fc
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title=None,
    title_fontsize=22,
    legend_fontsize=16,
    height=2,
)

### Relative aggregate macro risk premium scores

In [ ]:
cidx = cids_fc
xcatx = xrpz + ["MARKETRISK_ZN", "FCBIXR_NSA", "FCBIXR_VT10"]

dfa = msp.make_relative_value(
    df = dfx,
    xcats = xcatx,
    cids = cidx,
    start="2000-01-01",
    blacklist=black_fc,
    postfix="vEM",
)

dfx = msm.update_df(dfx, dfa)

xrpzr = [xcat + "vEM" for xcat in xrpz]

dict_labels["MACROSPREAD_RPS_ZNvEM"] = "Relative macro-spread risk premium score"
dict_labels["MACRORATING_RPS_ZNvEM"] = "Relative macro-rating risk premium score"
dict_labels["MACROALL_RPS_ZNvEM"] = "Relative macro risk premium score"

dict_labels["MARKETRISK_ZNvEM"] = "Relative market risk score"

dict_labels["MACROSPREAD_RPS_CWS_ZNvEM"] = "Relative macro-spread risk premium score, custom weights"
dict_labels["MACRORATING_RPS_CWS_ZNvEM"] = "Relative macro-rating risk premium score, custom weights"
dict_labels["MACROALL_RPS_CWS_ZNvEM"] = "Relative macro risk premium score, custom weights"

In [ ]:
cidx = cids_fc
xcatx = [rp + "vEM" for rp in rpz]
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
    xcat_labels=dict_labels,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    aspect=2,
    xcat_labels=dict_labels,
    cid_labels=dict_countries,
    title="Relative macro risk premium scores (all versus 24 EM average)",
    title_fontsize=25,
    legend_fontsize=16,
    height=2,
)

In [ ]:
cidx = cids_fc
xcatx = ["LTFCRATING_INV_ZN"]
sdate = "2000-01-01"

msp.view_ranges(
    dfx,
    xcats=xcatx,
    kind="bar",
    sort_cids_by=None,
    start=sdate,
)

msp.view_timelines(
    dfx,
    xcats=xcatx,
    cids=cidx,
    ncol=4,
    start=sdate,
    same_y=True,
    cid_labels=dict_countries,
    aspect=2,
    title='Long term credit rating, inverted',
    title_fontsize=22,
    legend_fontsize=16,
    height=2,
)

### Relative conceptual macro risk premium scores

In [ ]:
cidx = cids_fc
xcatx = crpz

dfa = msp.make_relative_value(
    df = dfx,
    xcats = xcatx,
    cids = cidx,
    start="2000-01-01",
    blacklist=black_fc,
    postfix="vEM",
)

dfx = msm.update_df(dfx, dfa)

crpzr = [xcat + "vEM" for xcat in crpz]

In [ ]:
dict_labels['GFINRISK_RPS_ZNvEM'] = 'Relative government finance conceptual macro risk premium score'
dict_labels['GOVRISK_RPS_ZNvEM'] = 'Relative governance conceptual macro risk premium score'
dict_labels['GROWTHRISK_RPS_ZNvEM'] = 'Relative growth risk conceptual macro risk premium score'
dict_labels['INFLRISK_RPS_ZNvEM'] =  'Relative inflation risk conceptual macro risk premium score'
dict_labels['XBALRISK_RPS_ZNvEM'] = 'Relative external balances conceptual macro risk premium score'
dict_labels['XDEBTRISK_RPS_ZNvEM'] = 'Relative foreign-currency debt conceptual macro risk premium score'
dict_labels['XLIABRISK_RPS_ZNvEM'] = 'Relative international position conceptual macro risk premium score'

## Value checks

### Composite directional signals

#### Specs and panel test

In [ ]:
dict_dir = {
    "sigs": ['MACROSPREAD_RPS_ZN','MACRORATING_RPS_ZN', 'MACROALL_RPS_ZN', 'MARKETRISK_ZN'],
    "targs": ["FCBIXR_VT10", "FCBIXR_NSA"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_dir

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["last", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=2,
    figsize=(14, 12),
    title="Risk scores and subsequent foreign currency bond index excess returns, 24 countries since 2000",
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=[dict_labels[key] for key in sigs]
)

In [ ]:
dix = dict_dir

sig = dix["sigs"][2]
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

cr = msp.CategoryRelations(
    dfx,
    xcats=[sig, ret],
    cids=cidx,
    freq="Q",
    years=None,
    lag=1,
    xcat_aggs=["last", "sum"],  # period mean adds stability
    start=start,
    blacklist=black,
)

cr.reg_scatter(
    title="Macro risk premia and subsequent foreign currency bond index excess returns, 24 countries since 2000",
    labels=False,
    xlab="Sovereign credit macro risk premium score, end of quarter",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    separator=2013,
    size=(10, 8),
)

cr.reg_scatter(
    title="Macro risk premia and subsequent foreign currency bond index excess returns, 24 countries since 2000",
    labels=False,
    xlab="Macro risk premium",
    ylab="Vol-adjusted returns",
    coef_box="lower right",
    prob_est="pool",
    separator="cids",
)

#### Accuracy and correlation check

In [ ]:
dix = dict_dir

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["last"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_dir
srr = dix["srr"]

tbl=srr.multiple_relations_table(signal_name_dict=dict_labels).round(3)
display(tbl.transpose())

In [ ]:
dix = dict_dir
srr = dix["srr"]

srr.accuracy_bars(type='cross_section', sigs="MACROALL_RPS_ZN", size=(16, 4))
srr.accuracy_bars(type='signals', size=(16, 4))

In [ ]:
dix = dict_dir
srr = dix["srr"]

srr.correlation_bars(type='cross_section', sigs="MACROALL_RPS_ZN", size=(16, 4))
srr.correlation_bars(type='signals', size=(16, 4), x_labels=dict_labels)

#### Naive PnLs

In [ ]:
dix = dict_dir

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

rt = ret.split('_')[-1]  # 'NSA' or 'VT10'

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    for bias in [0, 1]:
        pnls.make_pnl(
            sig=sig,
            sig_op="zn_score_pan",
            thresh=2,
            sig_add = bias,
            rebal_freq="monthly",
            neutral="zero",
            pnl_name=sig + "_PNL" + rt + str(bias),
            rebal_slip=1,
            vol_scale=10,
    )
pnls.make_long_pnl(vol_scale=10, label="Long only")

dix["pnls_"+rt.lower()] = pnls

In [ ]:
dix = dict_dir
rt = "VT10"
bias = 1
pnls = dix["pnls_"+rt.lower()] 

sigs = ["MACROALL_RPS_ZN", "MARKETRISK_ZN"]
strats = [sig + "_PNL" + rt + str(bias) for sig in sigs]

pnl_labels = {
    "MACROALL_RPS_ZN_PNL" + rt + str(bias): "Macro risk premium score with long bias",
    "MARKETRISK_ZN_PNL" + rt + str(bias): "Market risk score with long bias",
    "Long only": "Long only risk parity",
}

pnls.plot_pnls(
    title="Naive PnL for risk-premia and benchmark strategies, 24 EM sovereigns, vol-targeted positions",
    pnl_cats=strats + ["Long only"],
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats + ["Long only"]).round(3))
pnls.signal_heatmap(pnl_name=f"MACROALL_RPS_ZN_PNL" + rt + str(bias), figsize=(20, 10))

In [ ]:
dix = dict_dir

sigs = dix['sigs']
ret = dix["targs"][1]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

rt = ret.split('_')[-1]  # 'NSA' or 'VT10'

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    for bias in [0, 1]:
        pnls.make_pnl(
            sig=sig,
            sig_op="zn_score_pan",
            thresh=2,
            sig_add = bias,
            rebal_freq="monthly",
            neutral="zero",
            pnl_name=sig + "_PNL" + rt + str(bias),
            rebal_slip=1,
            vol_scale=10,
    )
pnls.make_long_pnl(vol_scale=10, label="Long only")

dix["pnls_"+rt.lower()] = pnls

In [ ]:
dix = dict_dir
rt = "NSA"
bias = 1
pnls = dix["pnls_"+rt.lower()] 

sigs = ["MACROALL_RPS_ZN", "MARKETRISK_ZN"]
strats = [sig + "_PNL" + rt + str(bias) for sig in sigs]

pnl_labels = {
    "MACROALL_RPS_ZN_PNL" + rt + str(bias): "Macro risk premium score with long bias",
    "MARKETRISK_ZN_PNL" + rt + str(bias): "Market risk score with long bias",
    "Long only": "Long only risk parity",
}

pnls.plot_pnls(
    title="Naive PnL for risk-premia and benchmark strategies, 24 EM sovereigns, nominal positions",
    pnl_cats=strats + ["Long only"],
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats + ["Long only"], label_dict=pnl_labels).round(3))
pnls.signal_heatmap(pnl_name=f"MACROALL_RPS_ZN_PNL" + rt + str(bias), figsize=(20, 10))

### Conceptual directional signals

#### Specs and panel test

In [ ]:
dict_cds = {
    "sigs": crpz + ['MARKETRISK_ZN'],
    "targs": ["FCBIXR_VT10", "FCBIXR_NSA"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_cds

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["mean", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=4,
    figsize=(14, 24),
    title="Spread-based premium scores and sovereign bond index returns, 24 EMBI countries, since 2000 or inception",
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=[dict_labels[key] for key in sigs]
)

#### Accuracy and correlation check

In [ ]:
dix = dict_cds

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["mean"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_cds
srr = dix["srr"]

tbl=srr.multiple_relations_table(signal_name_dict=dict_labels).round(3)
display(tbl.transpose())

In [ ]:
dix = dict_cds
srr = dix["srr"]

sig = dix['sigs'][0]
srr.accuracy_bars(type='cross_section', sigs=sig, size=(16, 4))
srr.accuracy_bars(type='signals', size=(16, 4), x_labels=dict_labels, x_labels_rotate= 60)

In [ ]:
dix = dict_cds
srr = dix["srr"]

sig = dix['sigs'][0]
srr.correlation_bars(type='cross_section', sigs=sig, size=(16, 4))
srr.correlation_bars(type='signals', size=(16, 4), x_labels=dict_labels,x_labels_rotate=70 )

#### Naive PnLs

In [ ]:
dix = dict_cds

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

rt = ret.split('_')[-1]  # 'NSA' or 'VT10'

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    for bias in [0, 1]:
        pnls.make_pnl(
            sig=sig,
            sig_op="zn_score_pan",
            thresh=2,
            sig_add = bias,
            rebal_freq="monthly",
            neutral="zero",
            pnl_name=sig + "_PNL" + rt + str(bias),
            rebal_slip=1,
            vol_scale=10,
    )
pnls.make_long_pnl(vol_scale=10, label="Long only")

dix["pnls_"+rt.lower()] = pnls

In [ ]:
dix = dict_cds
rt = "VT10"
bias = 1
pnls = dix["pnls_"+rt.lower()] 

sigs = dix["sigs"]
strats = [sig + "_PNL" + rt + str(bias) for sig in sigs]

pnl_labels = {
    'GFINRISK_RPS_ZN_PNLVT101': "Government finance conceptual macro risk premium score",
    'XBALRISK_RPS_ZN_PNLVT101': "External balances conceptual macro risk premium score",
    'XLIABRISK_RPS_ZN_PNLVT101': "International position conceptual macro risk premium score",
    'XDEBTRISK_RPS_ZN_PNLVT101': "Foreign-currency debt conceptual macro risk premium score",
    'GOVRISK_RPS_ZN_PNLVT101': "Governance conceptual macro risk premium score",
    'GROWTHRISK_RPS_ZN_PNLVT101': "Growth risk conceptual macro risk premium score",
    'INFLRISK_RPS_ZN_PNLVT101': "Inflation risk conceptual macro risk premium score",
    'MARKETRISK_ZN_PNLVT101': "Market risk score",
    'Long only': 'Long only'
}

pnls.plot_pnls(
    title=None,
    pnl_cats=strats + ["Long only"],
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats + ["Long only"], label_dict=pnl_labels).round(3))

### Composite relative signals

#### Specs and panel test

In [ ]:
dict_rel = {
    "sigs": ['MACROSPREAD_RPS_ZNvEM','MACRORATING_RPS_ZNvEM', 'MACROALL_RPS_ZNvEM', 'MARKETRISK_ZNvEM'],
    "targs": ["FCBIXR_VT10vEM", "FCBIXR_NSAvEM"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_rel

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["mean", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=2,
    figsize=(14, 12),
    title="Relative risk scores and subsequent relative excess returns, 24 EM sovereigns since 2000",
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index relative excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=[dict_labels[key] for key in sigs]
)

In [ ]:
dix = dict_rel

sig = dix["sigs"][2]
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

cr = msp.CategoryRelations(
    dfx,
    xcats=[sig, ret],
    cids=cidx,
    freq="Q",
    years=None,
    lag=1,
    xcat_aggs=["mean", "sum"],  # period mean adds stability
    start=start,
    blacklist=black,
)

cr.reg_scatter(
    title="Relative macro risk premia and subsequent foreign currency bond index excess returns, 24 countries since 2000",
    labels=False,
    xlab="Relative sovereign credit macro risk premium score, end of quarter",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    separator=2012,
    size=(12, 8),
)

cr.reg_scatter(
    title="Relative macro risk premia and subsequent foreign currency bond index excess returns, 24 countries since 2000",
    labels=False,
    xlab="Relative macro risk premium",
    ylab="Returns",
    coef_box="lower right",
    prob_est="pool",
    separator="cids",
)

#### Accuracy and correlation check

In [ ]:
dix = dict_rel

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["mean"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_rel
srr = dix["srr"]

tbl=srr.multiple_relations_table(signal_name_dict=dict_labels, return_name_dict={'FCBIXR_VT10vEM': 'Relative foreign currency bond index returns, volatility adjusted'}).round(3)
display(tbl.transpose())

In [ ]:
dix = dict_rel
srr = dix["srr"]

srr.accuracy_bars(type='cross_section', sigs="MACROALL_RPS_ZNvEM", size=(16, 4))
srr.accuracy_bars(type='signals', size=(16, 4), x_labels=dict_labels, x_labels_rotate= 60)

In [ ]:
dix = dict_rel
srr = dix["srr"]

srr.correlation_bars(type='cross_section', sigs="MACROALL_RPS_ZNvEM", size=(16, 4))
srr.correlation_bars(type='signals', size=(16, 4),x_labels=dict_labels, x_labels_rotate= 60)

#### Naive PnLs

In [ ]:
dix = dict_rel

sigs = dix["sigs"]
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    pnls.make_pnl(
        sig=sig,
        sig_op="zn_score_pan",
        thresh=2,
        sig_add=0,
        rebal_freq="monthly",
        neutral="zero",
        pnl_name=sig + "_PNL",
        rebal_slip=1,
        vol_scale=10,
    )

dix["pnls"] = pnls

In [ ]:
dix = dict_rel
bias = 0
pnls = dix["pnls"]
sigs = dix["sigs"]

sigs = ["MACROSPREAD_RPS_ZNvEM", "MACRORATING_RPS_ZNvEM", "MACROALL_RPS_ZNvEM"]
strats = [sig + "_PNL" for sig in sigs]

pnl_labels = {
    "MACROSPREAD_RPS_ZNvEM_PNL": "Spread-based relative macro risk premium score",
    "MACRORATING_RPS_ZNvEM_PNL": "Ratings-based relative macro risk premium score",
    "MACROALL_RPS_ZNvEM_PNL": "Composite relative macro risk premium score",
}

pnls.plot_pnls(
    title="Naive PnL for relative risk-premia strategies, 24 EM sovereigns, 7-factor macro risk",
    pnl_cats=strats,
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats).round(3))
pnls.signal_heatmap(
    pnl_name=f"MACROALL_RPS_ZNvEM_PNL",
    figsize=(16, 6),
    title="Signal heatmap for relative macro risk premium score, 24 EM sovereigns, 7-factor macro risk",
    tick_fontsize=10,
)

### Conceptual relative signals

#### Specs and panel test

In [ ]:
dict_crs = {
    "sigs": crpzr + ["MARKETRISK_ZNvEM"],
    "targs": ["FCBIXR_VT10vEM", "FCBIXR_NSAvEM"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_crs

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["mean", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=4,
    figsize=(14, 24),
    title=None,
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=['Relative ' + dict_labels[key[:-3]] for key in sigs]
)

#### Accuracy and correlation check

In [ ]:
dix = dict_crs

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["mean"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_crs
srr = dix["srr"]
tbl=srr.multiple_relations_table(signal_name_dict=dict_labels,return_name_dict={'FCBIXR_VT10vEM': 'Relative foreign currency bond index returns, volatility adjusted'}).round(3)
display(tbl.transpose())

#### Naive PnLs

In [ ]:
dix = dict_crs

sigs = dix["sigs"]
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    pnls.make_pnl(
        sig=sig,
        sig_op="zn_score_pan",
        thresh=2,
        sig_add=0,
        rebal_freq="monthly",
        neutral="zero",
        pnl_name=sig + "_PNL",
        rebal_slip=1,
        vol_scale=10,
    )

dix["pnls"] = pnls


In [ ]:
dix = dict_crs
bias = 0
pnls = dix["pnls"] 

sigs = dix["sigs"]
strats = [sig + "_PNL" for sig in sigs]

pnl_labels = {}

for sig in sigs:
    pnl_labels[sig+'_PNL'] = dict_labels[sig]
    

pnls.plot_pnls(
    title=None,
    pnl_cats=strats,
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats, label_dict=pnl_labels).round(3))

### Customized directional signals

#### Specs and panel test

In [ ]:
dict_dirc = {
    "sigs": ['MACROSPREAD_RPS_CWS_ZN','MACRORATING_RPS_CWS_ZN', 'MACROALL_RPS_CWS_ZN', 'MARKETRISK_ZN'],
    "targs": ["FCBIXR_VT10", "FCBIXR_NSA"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_dirc

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["last", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=2,
    figsize=(14, 12),
    title=None,
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=[dict_labels[key] for key in sigs]
)

#### Accuracy and correlation check

In [ ]:
dix = dict_dirc

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["last"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_dirc
srr = dix["srr"]

tbl=srr.multiple_relations_table(signal_name_dict=dict_labels,return_name_dict={'FCBIXR_VT10vEM': 'Relative foreign currency bond index returns, volatility adjusted'}).round(3)
display(tbl.transpose())

#### Naive PnLs

In [ ]:
dix = dict_dirc

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

rt = ret.split('_')[-1]  # 'NSA' or 'VT10'

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    for bias in [0, 1]:
        pnls.make_pnl(
            sig=sig,
            sig_op="zn_score_pan",
            thresh=2,
            sig_add = bias,
            rebal_freq="monthly",
            neutral="zero",
            pnl_name=sig + "_PNL" + rt + str(bias),
            rebal_slip=1,
            vol_scale=10,
    )
pnls.make_long_pnl(vol_scale=10, label="Long only")

dix["pnls_"+rt.lower()] = pnls

In [ ]:
dix = dict_dirc
rt = "VT10"
bias = 1
pnls = dix["pnls_"+rt.lower()] 

sigs = ["MACROALL_RPS_CWS_ZN", "MARKETRISK_ZN"]
strats = [sig + "_PNL" + rt + str(bias) for sig in sigs]

pnl_labels = {
    'MACROALL_RPS_CWS_ZN_PNLVT101': 'Macro risk premium customised directional score, vol targeted returns',
    'MARKETRISK_ZN_PNLVT101': 'Market risk score, directional, vol targeted returns',
    'Long only': 'Long only'
}

pnls.plot_pnls(
    title=None,
    pnl_cats=strats + ["Long only"],
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats + ["Long only"], label_dict=pnl_labels).round(3))
pnls.signal_heatmap(pnl_name=f"MACROALL_RPS_CWS_ZN_PNL" + rt + str(bias), figsize=(20, 10))

In [ ]:
dix = dict_dirc

sigs = dix['sigs']
ret = dix["targs"][1]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

rt = ret.split('_')[-1]  # 'NSA' or 'VT10'

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    for bias in [0, 1]:
        pnls.make_pnl(
            sig=sig,
            sig_op="zn_score_pan",
            thresh=2,
            sig_add = bias,
            rebal_freq="monthly",
            neutral="zero",
            pnl_name=sig + "_PNL" + rt + str(bias),
            rebal_slip=1,
            vol_scale=10,
    )
pnls.make_long_pnl(vol_scale=10, label="Long only")

dix["pnls_"+rt.lower()] = pnls

In [ ]:
dix = dict_dirc
rt = "NSA"
bias = 1
pnls = dix["pnls_"+rt.lower()] 

sigs = ["MACROALL_RPS_CWS_ZN", "MARKETRISK_ZN"]
strats = [sig + "_PNL" + rt + str(bias) for sig in sigs]

pnl_labels = {
    'MACROALL_RPS_CWS_ZN_PNLNSA1': 'Macro risk premium customised directional score',
    'MARKETRISK_ZN_PNLNSA1': 'Market risk score, directional',
    'Long only': 'Long only'
}
pnls.plot_pnls(
    title=None,
    pnl_cats=strats + ["Long only"],
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats + ["Long only"], label_dict=pnl_labels).round(3))
pnls.signal_heatmap(pnl_name=f"MACROALL_RPS_CWS_ZN_PNL" + rt + str(bias), figsize=(20, 10))

### Customized relative signals

#### Specs and panel test

In [ ]:
dict_relc = {
    "sigs": ['MACROSPREAD_RPS_CWS_ZNvEM','MACRORATING_RPS_CWS_ZNvEM', 'MACROALL_RPS_CWS_ZNvEM', 'MARKETRISK_ZNvEM'],
    "targs": ["FCBIXR_VT10vEM", "FCBIXR_NSAvEM"],
    "cidx": cids_fc,
    "start": "2000-01-01",
    "black": black_fc,
}

In [ ]:
dix = dict_relc

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

catregs = {}
for sig in sigs:
    catregs[sig] = msp.CategoryRelations(
        dfx,
        xcats=[sig, ret],
        cids=cidx,
        freq="Q",
        lag=1,
        xcat_aggs=["mean", "sum"],
        start=start,
        blacklist=black,
    )

msv.multiple_reg_scatter(
    cat_rels=[v for k, v in catregs.items()],
    ncol=2,
    nrow=2,
    figsize=(14, 12),
    title=None,
    title_fontsize=20,
    xlab="End-of-quarter score",
    ylab="Foreign currency bond index excess returns, vol targeted, %, next quarter",
    coef_box="lower right",
    prob_est="map",
    single_chart=True,
    subplot_titles=[dict_labels[key] for key in sigs]
)

#### Accuracy and correlation check

In [ ]:
dix = dict_relc

sigs = dix['sigs']
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

srr = mss.SignalReturnRelations(
    dfx,
    rets=ret,
    cids=cidx,
    sigs=sigs,
    # cosp=True,
    freqs="M",
    agg_sigs=["mean"],  # for stability
    start=start,
)

dix["srr"] = srr

In [ ]:
dix = dict_relc
srr = dix["srr"]

tbl=srr.multiple_relations_table(signal_name_dict=dict_labels,return_name_dict={'FCBIXR_VT10vEM': 'Relative foreign currency bond index returns, volatility adjusted'}).round(3)
display(tbl.transpose())

#### Naive PnLs

In [ ]:
dix = dict_relc

sigs = dix["sigs"]
ret = dix["targs"][0]
cidx = dix["cidx"]
start = dix["start"]
black = dix["black"]

pnls = msn.NaivePnL(
    df=dfx,
    ret=ret,
    sigs=sigs,
    cids=cidx,
    start=start,
    blacklist=black,
    bms=bms,
)
for sig in sigs:
    pnls.make_pnl(
        sig=sig,
        sig_op="zn_score_pan",
        thresh=2,
        sig_add=0,
        rebal_freq="monthly",
        neutral="zero",
        pnl_name=sig + "_PNL",
        rebal_slip=1,
        vol_scale=10,
    )

dix["pnls"] = pnls

In [ ]:
dix = dict_relc
bias = 0
pnls = dix["pnls"]
sigs = dix["sigs"]

sigs = ["MACROSPREAD_RPS_CWS_ZNvEM", "MACRORATING_RPS_CWS_ZNvEM", "MACROALL_RPS_CWS_ZNvEM"]
strats = [sig + "_PNL" for sig in sigs]

pnl_labels = {
    "MACROSPREAD_RPS_CWS_ZNvEM_PNL": "Spread-based relative macro risk premium score",
    "MACRORATING_RPS_CWS_ZNvEM_PNL": "Ratings-based relative macro risk premium score",
    "MACROALL_RPS_CWS_ZNvEM_PNL": "Composite relative macro risk premium score",
}

pnls.plot_pnls(
    title="Naive PnL for relative risk-premia strategies, 24 EM sovereigns, 4-factor macro risk",
    pnl_cats=strats,
    xcat_labels=pnl_labels,
    title_fontsize=14,
)
display(pnls.evaluate_pnls(pnl_cats=strats,label_dict=pnl_labels).round(3))
pnls.signal_heatmap(pnl_name=f"MACROALL_RPS_CWS_ZNvEM_PNL", figsize=(20, 10))